# Phase 2c (Kaggle): fill the gaps in the results tables

Two training runs the README tables are missing:

| Run | Why it's missing | Expected time on a T4 |
|---|---|---|
| **bicycle, mcmc 1M, 30k** | second Mip-NeRF 360 scene, never run | ~35-45 min |
| **garden, default, 30k** | ran out of GPU memory on Colab at ~11.8k steps (`docs/failures.md` F2.1) | ~2.5 h, if it fits |

**Why Kaggle:** a committed Kaggle notebook keeps running with the browser closed (up to 12 h), and its GPU quota (30 h/week) is separate from Colab's. Its free GPUs are also 16 GB, so Kaggle by itself doesn't fix F2.1. What might: the Colab run crashed with **4.19 GB reserved but unused** (fragmentation) *before* `expandable_segments` was added, and gsplat's `--packed` mode uses less memory without changing the result. If it still runs out of memory, that's a firm answer: default ADC on garden needs more than 16 GB.

**How to run:**
1. Kaggle → *Create → New Notebook → File → Import Notebook* → this file.
2. Right panel → *Settings*: **Accelerator: GPU T4 x2** (not P100: recent PyTorch builds no longer support it), **Internet: on** (needs a phone-verified account).
3. *Save Version → Save & Run All (Commit)*. You can close the tab. It runs in order: install (~15-20 min, gsplat compiles once), bicycle, then garden default.
4. When it finishes: open the version → **Output**: `kaggle_results.md` is the table to paste back; grids are in `figures/`.

Only one GPU is used: gsplat's trainer runs one process per scene.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, re, subprocess, shutil
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)

# (scene, experiment name, trainer arguments). Runs in this order; a failed run doesn't stop the next one.
RUNS = [
    ('bicycle', 'mcmc_1M', ['mcmc', '--strategy.cap-max', '1000000']),
    ('garden',  'default', ['default', '--packed']),     # --packed: same result, less memory (F2.1)
]
STEPS = 30_000

# Published 3DGS (Kerbl et al. 2023, per-scene table, 30k steps, 1/4 res). Check before quoting.
PUBLISHED = {'garden': {'psnr': 27.41, 'ssim': 0.868, 'lpips': 0.103},
             'bicycle': {'psnr': 25.25, 'ssim': 0.771, 'lpips': 0.205}}

PROJECT = '/kaggle/working'          # saved as the notebook's Output when the commit finishes
WORK    = '/kaggle/tmp'              # scratch, not saved
FIG     = f'{PROJECT}/figures'
for d in [f'{PROJECT}/results', f'{PROJECT}/cache', WORK, FIG]:
    os.makedirs(d, exist_ok=True)
run_name = lambda scene, exp: f'phase2_{scene}_1of4_{exp}_{STEPS // 1000}k'

# fail fast if Kaggle's internet is off (pip and the dataset download both need it)
import socket
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close()
    print('internet: ok')
except OSError:
    raise SystemExit('❌ No internet. Kaggle: right panel → Settings → Internet → On '
                     '(needs a phone-verified account: kaggle.com → Settings → Phone verification). Then run again.')

## 2. Install gsplat

Same steps as the Colab notebooks, with two Kaggle changes: nothing restarts the kernel (a restart would abort a committed run), and numpy is left as Kaggle ships it.

In [ ]:
import importlib.util
MARKER = '/tmp/.gs_deps_ok'

if not os.path.exists(MARKER):
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:   # harmless on numpy 1, required on numpy 2
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
os.environ['GSPLAT_PREBUILT_DIR'] = '/kaggle/tmp/gsplat_prebuilt'
# F1.5 fix, Kaggle version: the compiled library is saved to the Output (cache/gsplat_jit/...).
# To skip the ~15 min compile next time: add that Output as a Kaggle Dataset and point JIT_CACHE at it.
import torch, importlib.util
PREBUILT = '/kaggle/tmp/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use $GSPLAT_PREBUILT_DIR/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/kaggle/tmp/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to the Output')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. Data: one Mip-NeRF 360 scene at a time (only `images_4`, poses, sparse model)

In [ ]:
from remotezip import RemoteZip
import imageio.v2 as imageio
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
WANT = ('sparse/', 'poses_bounds.npy', 'images_4/')

def get_scene(scene):
    data = f'{WORK}/data/360_v2/{scene}'
    if not os.path.exists(f'{data}/images_4'):
        t = time.time()
        with RemoteZip(URL) as z:
            members = [m for m in z.namelist() if m.startswith(f'{scene}/') and not m.endswith('/')
                       and m.split('/', 1)[1].startswith(WANT)]
            for m in members:
                dst = f'{data}/' + m.split('/', 1)[1]
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(m) as src_, open(dst, 'wb') as f:
                    f.write(src_.read())
        print(f'{scene}: downloaded {len(members)} files in {time.time()-t:.0f}s')
    if not os.path.exists(f'{data}/images'):
        os.symlink(f'{data}/images_4', f'{data}/images')      # train the provided 1/4 images with --data_factor 1 (F1.2)
    print(scene, len(os.listdir(f'{data}/images')), 'images | size', imageio.imread(glob.glob(f'{data}/images/*')[0]).shape)
    return data

## 4. Train

Each run saves its scores at 7k and 30k. If a run fails (e.g. out of memory), its log is kept and the next run still starts.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
assert re.search(r'lpips.net', help_txt), 'this gsplat has no --lpips_net option'

def step_of(path):
    return int(re.search(r'step(\d+)', os.path.basename(path)).group(1))

def stats_files(rd, stage):
    return sorted(glob.glob(f'{rd}/stats/{stage}_step*.json'), key=step_of)    # numeric order: 6999 before 29999

def check_resolution(rd, data_dir):
    expected = imageio.imread(sorted(glob.glob(f'{data_dir}/images/*'))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2   # GT | render
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')
    return int(actual)

def train(scene, exp, args, data):
    rd = f'{PROJECT}/results/{run_name(scene, exp)}'
    done = [f for f in stats_files(rd, 'val') if step_of(f) == STEPS - 1]
    if done:
        print(f'✔ {run_name(scene, exp)}: already trained'); return
    if os.path.exists(rd):                      # a half-finished run can't be resumed: keep its files, start fresh
        shutil.move(rd, rd + time.strftime('_failed_%Y%m%d_%H%M'))
    cmd = [sys.executable, 'simple_trainer.py', *args, '--data_dir', data, '--data_factor', '1',
           '--result_dir', rd, '--max_steps', str(STEPS), '--disable_viewer', '--lpips_net', 'vgg']
    os.makedirs(rd, exist_ok=True)
    print('$', ' '.join(cmd[1:]))
    t0 = last = time.time(); buf = ''; tail = []
    env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}   # less memory fragmentation (F2.1)
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if 'PSNR' in ln or time.time() - last > 120:
                    print(f'[{(time.time()-t0)/60:5.1f} min] {ln.strip()[:150]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    json.dump({'wall_min': round((time.time() - t0) / 60, 1)}, open(f'{rd}/wall_time.json', 'w'))
    print(f'✔ {run_name(scene, exp)} finished in {(time.time()-t0)/60:.1f} min | trained at width {check_resolution(rd, data)}px')

STATUS = {}
for scene, exp, args in RUNS:
    try:
        train(scene, exp, args, get_scene(scene)); STATUS[(scene, exp)] = 'finished'
    except Exception as e:                       # keep going: the next run may still fit
        STATUS[(scene, exp)] = f'failed: {str(e)[:200]}'
        print(f'❌ {scene} {exp}: {e}')
    import gc; gc.collect()
print(STATUS)

## 5. Results (also written to `kaggle_results.md` in the Output)

In [ ]:
import numpy as np, torch
BYTES_PER_GS, SPLAT_BYTES = 62 * 4, 32       # .ply (SH degree 3) and web .splat sizes per Gaussian

def oom_step(rd):
    log = f'{rd}/train_log.txt'
    if not os.path.exists(log): return None
    txt = open(log, errors='ignore').read()
    if 'OutOfMemoryError' not in txt: return None
    steps = re.findall(r'(\d+)/\d+ \[', txt)
    return int(steps[-1]) if steps else '?'

lines = ['| Scene | Strategy | Steps | #Gaussians | PSNR | SSIM | LPIPS (vgg) | vs paper (PSNR) | Train | Peak VRAM | Render FPS | .ply / .splat |',
         '|---|---|---|---|---|---|---|---|---|---|---|---|']
gpu = torch.cuda.get_device_name(0)
for scene, exp, args in RUNS:
    rd = f'{PROJECT}/results/{run_name(scene, exp)}'
    vals = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'val')}
    trains = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'train')}
    for step in sorted(vals):
        v, tr = vals[step], trains.get(step, {})
        pub = PUBLISHED[scene]
        gap = f"{v['psnr'] - pub['psnr']:+.2f} dB" if step == STEPS - 1 else '(7k)'
        lines.append(f"| {scene} | {' '.join(args)} | {step + 1:,} | {v['num_GS']:,} | {v['psnr']:.2f} | {v['ssim']:.3f} | "
                     f"{v['lpips']:.3f} | {gap} | {tr.get('ellipse_time', float('nan')) / 60:.1f} min | "
                     f"{tr.get('mem', float('nan')):.1f} GB | {1 / v['ellipse_time']:.0f} | "
                     f"{v['num_GS'] * BYTES_PER_GS / 1e6:.0f} MB / {v['num_GS'] * SPLAT_BYTES / 1e6:.0f} MB |")
    if not any(s == STEPS - 1 for s in vals):
        o = oom_step(rd)
        lines.append(f"| {scene} | {' '.join(args)} | 30,000 | – | " +
                     (f"*out of memory at step ~{o}* |" if o else f"*{STATUS.get((scene, exp), 'not run')}* |") +
                     " – | – | – | – | – | – | – |")
lines.append(f'\nGPU: {gpu} · published 3DGS (30k): ' +
             ' · '.join(f"{s} {p['psnr']} / {p['ssim']} / {p['lpips']}" for s, p in PUBLISHED.items()))
table = '\n'.join(lines)
open(f'{PROJECT}/kaggle_results.md', 'w').write(table + '\n')
print(table)

## 6. Render grids: real | render | error

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
VIEWS = [0, 8, 16]
for scene, exp, args in RUNS:
    rd = f'{PROJECT}/results/{run_name(scene, exp)}'
    files = [f'{rd}/renders/val_step{STEPS - 1}_{v:04d}.png' for v in VIEWS]
    if not all(os.path.exists(f) for f in files):
        continue
    fig, ax = plt.subplots(len(VIEWS), 3, figsize=(18, 4 * len(VIEWS)))
    for i, f in enumerate(files):
        im = imageio.imread(f)[..., :3].astype(np.float32) / 255
        w = im.shape[1] // 2; gt, pr = im[:, :w], im[:, w:]
        ax[i, 0].imshow(gt); ax[i, 0].set_title(f'real (test view {VIEWS[i]})')
        ax[i, 1].imshow(pr); ax[i, 1].set_title(f'{scene} {exp}, PSNR {-10 * np.log10(((gt - pr) ** 2).mean()):.2f}')
        ax[i, 2].imshow(np.abs(gt - pr).mean(-1), cmap='inferno', vmin=0, vmax=0.25); ax[i, 2].set_title('error')
    for a in ax.flat: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/grid_{scene}_{exp}.jpg', dpi=80); plt.close()
    print('saved', f'{FIG}/grid_{scene}_{exp}.jpg')

# checkpoints are large (default ADC: >1 GB). Keep only the 30k one so the Output stays small.
for f in glob.glob(f'{PROJECT}/results/*/ckpts/ckpt_*_rank0.pt'):
    if not f.endswith(f'ckpt_{STEPS - 1}_rank0.pt'):
        os.remove(f)
print(subprocess.run(['du', '-sh', PROJECT], capture_output=True, text=True).stdout)

## ✅ Checklist
- [ ] `kaggle_results.md` in the Output: paste it into the chat
- [ ] Did garden default finish? If it ran out of memory again, note the step: F2.1 then has its answer (needs > 16 GB)
- [ ] Optional: download `figures/grid_*.jpg`